In [1]:
import os
from pathlib import Path
import json

from conch.open_clip_custom import create_model_from_pretrained
from conch.downstream.zeroshot_path import zero_shot_classifier, run_zeroshot

import torch
import torch.nn.functional as F
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# display all jupyter output
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

c:\Users\datainsight\anaconda3\envs\exaonepath\lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
root = Path('../').resolve()
os.chdir(root)

This notebook provides a complete example for performing zero-shot classification by ensembling multiple prompts and prompt templates. You can use this notebook to reproduce the zero-shot classification results on CRC100K (image size = 224 x 224). 

In [3]:
model_cfg = 'conch_ViT-B-16'
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
checkpoint_path = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_aggregation\caption_generation\checkpoints\CONCH\pytorch_model.bin"
force_image_size = 224
model, preprocess = create_model_from_pretrained(model_cfg, checkpoint_path, device=device,
                                                 force_image_size=force_image_size)
_ = model.eval()

In [11]:
data_source = r"D:\Aamir Gulzar\dataset\CRC100K\NCT-CRC-HE-100K-NONORM"
dataset = ImageFolder(data_source, transform=preprocess)
dataloader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=4)
if hasattr(dataloader.dataset, 'class_to_idx'):
     idx_to_class = {v:k for k,v in dataloader.dataset.class_to_idx.items()}
else:
     raise ValueError('Dataset does not have label_map attribute')
print("num samples: ", len(dataloader.dataset))
print(idx_to_class)

num samples:  100000
{0: 'ADI', 1: 'BACK', 2: 'DEB', 3: 'LYM', 4: 'MUC', 5: 'MUS', 6: 'NORM', 7: 'STR', 8: 'TUM'}


In [12]:
prompt_file = "D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_aggregation\caption_generation\config_files\CRC100K_prompts_all_per_class.json"
with open(prompt_file) as f:
    prompts = json.load(f)['0']
classnames = prompts['classnames']
templates = prompts['templates']
n_classes = len(classnames)
classnames_text = [classnames[str(idx_to_class[idx])] for idx in range(n_classes)]
for class_idx, classname in enumerate(classnames_text):
    print(f'{class_idx}: {classname}')

0: ['adipose', 'adipose tissue', 'adipocytes', 'fat', 'fat cells']
1: ['background', 'background tissue', 'slide background', 'non-tissue region']
2: ['debris', 'necrosis', 'necrotic tissue', 'necrotic debris', 'tissue necrosis']
3: ['lymphocytes', 'immune cells', 'lymphoid cells', 'white blood cells', 'lymphoid infiltrate']
4: ['mucus', 'mucin', 'mucus pool', 'mucin pool']
5: ['smooth muscle', 'smooth muscle tissue', 'muscle', 'muscularis propria', 'muscularis mucosa']
6: ['normal mucosa', 'normal colon mucosa', 'benign mucosa', 'uninvolved mucosa']
7: ['tumor-associated stroma', 'cancer-associated stroma', 'stromal cells', 'stromal tissue', 'stroma']
8: ['tumor', 'tumor cells', 'malignant cells', 'carcinoma']


In [13]:
zeroshot_weights = zero_shot_classifier(model, classnames_text, templates, device=device)
print(zeroshot_weights.shape)

torch.Size([512, 9])


In [14]:
results, dump = run_zeroshot(model, zeroshot_weights, dataloader, device, 
                    dump_results=True, metrics=['bacc', 'weighted_f1'])

100%|██████████| 1563/1563 [06:48<00:00,  3.83it/s]


In [15]:
for k, v in results.items():
    print(f'{k}: {v:.3f}')

bacc: 0.749
weighted_f1: 0.737
